Wikidata Preparation Notebook
Project: Distractor Filtering Through Open KG Concept Proximity for Dense RAG

This notebook covers Wikipedia corpus exploration and passage segmentation


In [1]:
# To use HuggingFace datasets with faster download speeds,
# place your read-only HF token in the .hf_token file at the project root.
# This file is gitignored and will not be committed.

from pathlib import Path
from huggingface_hub import login

token_path = Path.cwd() / ".hf_token"
if token_path.exists():
    token = token_path.read_text().strip()
    if token and token != "YOUR_TOKEN_HERE":
        login(token=token)
        print("Logged in to HuggingFace Hub")
    else:
        print("No token set — put your HF token in .hf_token")
else:
    print("No .hf_token file found — running unauthenticated (slower downloads)")

Logged in to HuggingFace Hub


## Step 0 — Configuration & Corpus Loading

**Corpus source**: `florin-hf/wiki_dump2018_nq_open` (HuggingFace) — the augmented Wikipedia corpus
from "The Power of Noise".

Based on the **Wikipedia Dec 20, 2018 dump** (same version used by DPR, Karpukhin et al. 2020),
this corpus integrates gold documents from the NQ dataset with duplicate filtering,
resulting in ~21M documents (columns: `text`, `title`).

We use this corpus directly because:
- It aligns with the experimental setup of our reference paper
- Gold documents are already embedded, enabling proper retrieval evaluation
- Articles are stored as full text — no need to reconstruct from DPR 100-word passages

**Local caching strategy**: on first run, the dataset is downloaded from HF and saved locally
as a TSV in `data/wikipedia_2018_clean/`. Subsequent runs load from disk (no re-download).

In [2]:
from pathlib import Path
from datasets import load_dataset
import pandas as pd

# --- Configuration ---
PROJECT_ROOT = Path.cwd()

# Local cache: once downloaded, the corpus is saved here as TSV
ARTICLES_PATH = PROJECT_ROOT / "data" / "wikipedia_2018_clean" / "articles_clean.tsv"

if ARTICLES_PATH.exists():
    print(f"Local cache found: {ARTICLES_PATH}")
    size_gb = ARTICLES_PATH.stat().st_size / (1024 ** 3)
    print(f"Size: {size_gb:.2f} GB")
else:
    print("Local cache not found — downloading from HuggingFace...")
    ds = load_dataset("florin-hf/wiki_dump2018_nq_open", split="train")
    print(f"Downloaded {len(ds):,} documents")

    # Save locally as TSV for fast reload
    ARTICLES_PATH.parent.mkdir(parents=True, exist_ok=True)
    ds.to_pandas().to_csv(ARTICLES_PATH, sep="\t", index=False)
    print(f"Saved to {ARTICLES_PATH}")
    del ds

Local cache found: C:\Users\Utente\Documents\PycharmProgetti\dl-RAG-denseAndKG\data\wikipedia_2018_clean\articles_clean.tsv
Size: 12.54 GB


## Step 1 — Sentence-aligned segmentation

### The problem: semantic bleeding
The DPR corpus cuts articles every 100 words **mechanically**, ignoring sentence boundaries.
A sentence can start in passage N and end in passage N+1. This causes:

1. **Entity linking degradation**: ReFiNed may fail to link entities in truncated sentences
2. **LLM reader confusion**: incomplete sentences at chunk boundaries add noise to Llama2's context

### Plan
Since the HF corpus already provides full articles, we only need to:
1. **Segment with sentence alignment**: apply a sentence-aware algorithm that keeps complete sentences within ~100 words
2. **Save as `psgs_w100_sentence.tsv`**: same TSV format (`id`, `text`, `title`), new passage boundaries

### Step 1a — Sentence-aligned segmentation algorithm

**Goal**: segment each article into passages of exactly 100 words with complete sentences.

**Algorithm (per article)**:
1. Split into sentences using regex (`(?<=[.!?])\s+`)
2. Greedy accumulation: add sentences to the current segment while `word_count < 100`
3. When the next sentence would push the count to ≥ 100, close the segment and start a new one
4. Sentences > 100 words: mechanical split at 100-word boundaries; if the final fragment is < 100 words, take the last 100 words of the sentence (backward slide, overlapping with the previous chunk)
5. Normal segments < 100 words: pad to 100 using words from the article's first segment

**Why pad from the first segment**: mirrors DPR's own strategy — gives every passage an explicit anchor to the article's identity (title + opening lines).

**Why backward slide instead of padding for long sentences**: preserves contextual contiguity. The overlap with the previous chunk is the accepted trade-off (sentences > 100 words are extremely rare in Wikipedia).

**Regex limitations**: false positives on abbreviations ("U.S. Army" → splits after "U.S."), but harmless — short fragments simply get merged into the same segment by the accumulator.

In [3]:
import re

# Pre-compiled regex for sentence splitting.
# Compiled once here in the notebook; in the .py module it's compiled
# once per worker at import time. Either way, avoids re-compilation
# on every segment_article call.
_SENTENCE_SPLIT = re.compile(r'(?<=[.!?])\s+')


def segment_article(text: str) -> list[str]:
    """Segment article text into 100-word passages with sentence alignment.

    Algorithm:
      1. Split text into sentences (regex on .!? followed by whitespace)
      2. Greedily pack sentences into segments while word_count < 100
      3. Sentences > 100 words: mechanical split at 100-word boundaries;
         final fragment < 100 words uses backward slide (last 100 words
         of the sentence) to preserve contextual contiguity
      4. Normal segments < 100 words: pad to exactly 100 words with words
         from the article's first segment (mirrors DPR padding strategy)

    Args:
        text: Full article text.

    Returns:
        List of passage strings, each exactly 100 words.
    """
    # Sentence split using pre-compiled regex.
    # Imperfect for abbreviations ("U.S. Army" splits after "U.S.") but
    # harmless: short fragments get merged into the same segment.
    sentences = _SENTENCE_SPLIT.split(text)

    # --- Greedy sentence accumulation ---
    # Pack sentences into segments, keeping word count strictly < 100.
    # When the next sentence would push the count to >= 100, close the
    # current segment and start a new one with that sentence.
    segments = []
    current = []
    current_wc = 0

    for sent in sentences:
        # count(' ') + 1: word count without allocating a list.
        # Equivalent to len(split(' ')) but O(1) memory (no list created).
        sent_wc = sent.count(' ') + 1

        if sent_wc >= 100:
            # --- Long sentence handler ---
            # Close any accumulated segment first
            if current:
                segments.append(' '.join(current))
                current = []
                current_wc = 0

            # Mechanical split at 100-word boundaries
            words = sent.split(' ')
            while len(words) >= 100:
                segments.append(' '.join(words[:100]))
                words = words[100:]

            # Final fragment < 100 words: backward slide.
            # Take the last 100 words of the original sentence to preserve
            # contextual contiguity (accepts overlap with previous chunk).
            if words:
                all_words = sent.split(' ')
                segments.append(' '.join(all_words[-100:]))
        elif current and current_wc + sent_wc >= 100:
            # Adding this sentence would reach/exceed 100 — close segment
            segments.append(' '.join(current))
            current = [sent]
            current_wc = sent_wc
        else:
            current.append(sent)
            current_wc += sent_wc

    # Flush remaining sentences as the last segment
    if current:
        segments.append(' '.join(current))

    if not segments:
        return []

    # --- Pad to exactly 100 words ---
    # Source: words from the first segment (same strategy as DPR, which pads
    # the last chunk from the article's beginning). Gives every passage an
    # explicit anchor to the article's identity.
    #
    # Circular repetition: if first_words has fewer words than padding_needed,
    # we cycle through them repeatedly. This guarantees exactly 100 words
    # even for very short articles (e.g., 30-word article needs 70 padding
    # words — cycles through the 30 words ~2.3 times).
    first_words = segments[0].split(' ')
    n_first = len(first_words)
    padded = []

    for seg in segments:
        n = seg.count(' ') + 1
        if n < 100:
            padding_needed = 100 - n
            # Modular indexing: word at position i comes from first_words[i % n_first]
            padding_words = [first_words[i % n_first] for i in range(padding_needed)]
            padded.append(seg + ' ' + ' '.join(padding_words))
        else:
            padded.append(seg)

    return padded


# --- Test on PAEEK (known article) ---
import polars as pl
paeek_row = pl.read_csv(ARTICLES_PATH, separator="\t").filter(pl.col("title") == "PAEEK")
paeek_text = paeek_row["text"][0]
print(f"PAEEK text: {paeek_text.count(' ') + 1} words\n")

segments = segment_article(paeek_text)
print(f"Segments: {len(segments)}\n")

for i, seg in enumerate(segments):
    wc = seg.count(' ') + 1
    words = seg.split(' ')
    start = ' '.join(words[:8])
    end = ' '.join(words[-8:])
    print(f"  [{i}] {wc} words")
    print(f"      start: {start} ...")
    print(f"      end:   ... {end}")
    print()

# --- Test circular padding on a short article ---
short_test = "This is a very short article with only twelve words in it."
short_segs = segment_article(short_test)
short_wc = short_segs[0].count(' ') + 1
print(f"Short article test ({short_test.count(' ') + 1} words):")
print(f"  Padded to: {short_wc} words (expected: 100)")
assert short_wc == 100, f"Circular padding failed! Got {short_wc}"
print(f"  Circular padding works")

PAEEK text: 100 words

Segments: 2

  [0] 100 words
      start: PAEEK PAEEK (; "Podosfairiki Athlitiki Enosi Eparxeias Kerynias" ...
      end:   ... PAEEK PAEEK (; "Podosfairiki Athlitiki Enosi Eparxeias Kerynias"

  [1] 100 words
      start: The PAEEK was a founding member of the ...
      end:   ... suspend it for some years up to date.

Short article test (12 words):
  Padded to: 100 words (expected: 100)
  Circular padding works


In [4]:
from pathlib import Path
import polars as pl

# --- File-based multiprocessing worker functions ---
# These functions are used by Pool workers for the file-based shared-nothing
# parallelization strategy. They read/write TSV files independently —
# only an integer (fragment index) travels through IPC.

# Worker-local globals, set by _init_file_worker via Pool initializer.
# Each spawned process calls the initializer once at startup, storing the
# paths in its own address space. The actual worker function then reads
# these globals — no need to pickle paths on every call.
_worker_input_dir: Path | None = None
_worker_output_dir: Path | None = None


def _init_file_worker(input_dir: str, output_dir: str) -> None:
    """Pool initializer: store I/O paths in worker-local globals."""
    global _worker_input_dir, _worker_output_dir
    _worker_input_dir = Path(input_dir)
    _worker_output_dir = Path(output_dir)


def file_segment_worker(frag_idx: int) -> int:
    """Process one input fragment, write output fragment.

    Reads input_dir/frag_{idx}.tsv (columns: title, text),
    applies segment_article to each article, writes flat passages
    to output_dir/frag_{idx}.tsv (columns: text, title).

    Resumability: if output file already exists, skip processing.

    Args:
        frag_idx: fragment index (0-99).

    Returns:
        Number of passages produced (-1 if skipped due to resumability).
    """
    input_path = _worker_input_dir / f"frag_{frag_idx}.tsv"
    output_path = _worker_output_dir / f"frag_{frag_idx}.tsv"

    # Resumability: skip if output already exists
    if output_path.exists():
        return -1

    df = pl.read_csv(input_path, separator="\t")

    titles = df["title"].to_list()
    texts = df["text"].to_list()

    # Build flat passage rows: one (text, title) per passage
    out_texts = []
    out_titles = []

    for title, text in zip(titles, texts):
        passages = segment_article(text)
        for passage in passages:
            out_texts.append(passage)
            out_titles.append(title)

    result_df = pl.DataFrame({
        "text": out_texts,
        "title": out_titles,
    })
    result_df.write_csv(output_path, separator="\t")

    return len(out_texts)


print("Worker functions defined: _init_file_worker, file_segment_worker")

Worker functions defined: _init_file_worker, file_segment_worker


In [5]:
import inspect

# === Write all functions to utils/text_processing.py ===
#
# Single source of truth: functions are DEFINED in notebook cells above,
# then written to the .py module here via inspect.getsource.
# On Windows, multiprocessing uses "spawn" — workers must IMPORT functions
# from a .py file (notebook-defined functions can't be pickled for import).
#
# This cell should be re-run whenever any function definition changes.

utils_dir = PROJECT_ROOT / "utils"
utils_dir.mkdir(exist_ok=True)
(utils_dir / "__init__.py").touch()

# --- Assemble module content ---
# Header: imports and module-level constants that functions reference.
# Functions: extracted verbatim via inspect.getsource.
MODULE_HEADER = '''\
import re
from pathlib import Path

import polars as pl


# Pre-compiled regex for sentence splitting (compiled once per worker at import).
_SENTENCE_SPLIT = re.compile(r'(?<=[.!?])\\s+')


# Worker-local globals, set by _init_file_worker via Pool initializer.
_worker_input_dir: Path | None = None
_worker_output_dir: Path | None = None

'''

module_content = MODULE_HEADER
module_content += inspect.getsource(segment_article) + "\n\n"
module_content += inspect.getsource(_init_file_worker) + "\n\n"
module_content += inspect.getsource(file_segment_worker) + "\n"

module_path = utils_dir / "text_processing.py"
module_path.write_text(module_content, encoding="utf-8")

print(f"Written to: {module_path}")
print(f"Functions: segment_article, _init_file_worker, file_segment_worker")
print(f"Size: {module_path.stat().st_size:,} bytes")

Written to: C:\Users\Utente\Documents\PycharmProgetti\dl-RAG-denseAndKG\utils\text_processing.py
Functions: segment_article, _init_file_worker, file_segment_worker
Size: 5,889 bytes


### Step 1b — Parallel segmentation (file-based shared-nothing)

**Problem**: applying `segment_article` to ~3.2M articles naively via `Pool.imap` requires
pickling ~11 GB of text to workers and ~11 GB of results back through OS pipes.
The pickle serialization/deserialization is single-threaded in the main process and
dominates wall-clock time.

**Solution**: file-based shared-nothing parallelism.
1. **Partition** the articles into 100 TSV fragments on disk
2. **Workers** receive only a fragment index (integer) via IPC — they read/write files independently
3. **Main process** concatenates output fragments and assigns sequential IDs

**IPC overhead**: ~400 bytes total (100 integers in + 100 integers out) instead of ~22 GB.

**Resumability**: if the process crashes, workers skip fragments whose output file already exists.

**Directory layout**:
```
data/wikipedia_2018_clean/ordered_fragments/       ← input (100 × ~32K articles)
data/wikipedia_2018_sentence_aligned/ordered_fragments/  ← output (100 × ~210K passages)
```

In [6]:
import polars as pl
from pathlib import Path
import time

# --- Resume guard: load articles if not already in memory ---
if "articles_df" not in dir():
    if "PROJECT_ROOT" not in dir():
        PROJECT_ROOT = Path.cwd()
    ARTICLES_PATH = PROJECT_ROOT / "data" / "wikipedia_2018_clean" / "articles_clean.tsv"
    print(f"Loading articles from {ARTICLES_PATH}...")
    articles_df = pl.read_csv(ARTICLES_PATH, separator="\t")
    print(f"Loaded {len(articles_df):,} articles")

if "PROJECT_ROOT" not in dir():
    PROJECT_ROOT = Path.cwd()

# --- Directory setup ---
INPUT_FRAG_DIR = PROJECT_ROOT / "data" / "wikipedia_2018_clean" / "ordered_fragments"
OUTPUT_FRAG_DIR = PROJECT_ROOT / "data" / "wikipedia_2018_sentence_aligned" / "ordered_fragments"

INPUT_FRAG_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FRAG_DIR.mkdir(parents=True, exist_ok=True)

# --- Partition articles into 100 fragments ---
# Each fragment is a TSV with columns (title, text) — only what the worker needs.
# The last fragment takes the remainder to avoid losing articles to integer division.

N_FRAGS = 100
n_articles = len(articles_df)
chunk_size = n_articles // N_FRAGS

print(f"Partitioning {n_articles:,} articles into {N_FRAGS} fragments")
print(f"  ~{chunk_size:,} articles per fragment")

# Skip if fragments already exist (resumability for the partitioning step too)
existing = list(INPUT_FRAG_DIR.glob("frag_*.tsv"))
if len(existing) == N_FRAGS:
    print(f"\n  All {N_FRAGS} input fragments already exist — skipping partitioning")
else:
    start = time.perf_counter()
    total_written = 0

    for i in range(N_FRAGS):
        offset = i * chunk_size
        # Last fragment takes all remaining articles
        length = chunk_size if i < N_FRAGS - 1 else n_articles - offset

        chunk = articles_df.slice(offset, length).select("title", "text")
        chunk.write_csv(INPUT_FRAG_DIR / f"frag_{i}.tsv", separator="\t")
        total_written += len(chunk)

    elapsed = time.perf_counter() - start
    print(f"\n  Written {total_written:,} articles in {elapsed:.1f}s")
    assert total_written == n_articles, f"Mismatch: {total_written} != {n_articles}"
    print(f"  Verification: {total_written:,} == {n_articles:,}")
    print(f"  Output: {INPUT_FRAG_DIR}")

Loading articles from C:\Users\Utente\Documents\PycharmProgetti\dl-RAG-denseAndKG\data\wikipedia_2018_clean\articles_clean.tsv...
Loaded 21,035,236 articles
Partitioning 21,035,236 articles into 100 fragments
  ~210,352 articles per fragment

  All 100 input fragments already exist — skipping partitioning


In [7]:
import sys
import time
import importlib
from multiprocessing import Pool, cpu_count
from tqdm import tqdm

# === File-based parallel segmentation ===
#
# Why file-based instead of Pool.imap with text data?
#   Pool.imap pickles every article text to workers and pickles results back.
#   For 3.2M articles (~11 GB text + ~11 GB results), the main process spends
#   most of its time in pickle.dumps/loads — single-threaded, on one core.
#
#   File-based: workers read/write files independently. The main process
#   sends only integer fragment indices (100 integers = ~400 bytes total IPC).
#   The actual text data flows through the filesystem, bypassing Python's
#   pickle serialization entirely.
#
# How it works:
#   1. Previous cell partitioned articles into 100 TSV files (input fragments)
#   2. Pool.imap distributes fragment indices [0..99] across workers
#   3. Each worker: reads its input TSV → segment_article per row → writes output TSV
#   4. Worker returns only the passage count (one integer back through the pipe)
#
# Resumability:
#   If the process crashes mid-way, workers skip fragments whose output file
#   already exists. Just re-run this cell to continue from where it stopped.
#
# Load balancing:
#   imap with 100 tasks on 12 workers: workers that finish early pick up the
#   next fragment. Better than static N-way split where one unlucky worker
#   gets all the long articles.

# Ensure project root is importable
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Reload to pick up latest changes to the module
import utils.text_processing
importlib.reload(utils.text_processing)
from utils.text_processing import _init_file_worker, file_segment_worker

n_cores = max(1, cpu_count())

print(f"Segmenting {N_FRAGS} fragments on {n_cores} workers")
print(f"  Input:  {INPUT_FRAG_DIR}")
print(f"  Output: {OUTPUT_FRAG_DIR}")

start = time.perf_counter()

# Pool initializer sets I/O paths in each worker's global state (once per process).
# The actual worker function receives only an integer — minimal IPC.
with Pool(
    n_cores,
    initializer=_init_file_worker,
    initargs=(str(INPUT_FRAG_DIR), str(OUTPUT_FRAG_DIR)),
) as pool:
    results = list(tqdm(
        pool.imap(file_segment_worker, range(N_FRAGS)),
        total=N_FRAGS,
        desc="Segmenting",
        unit="frag",
    ))

elapsed = time.perf_counter() - start

# --- Summary ---
skipped = sum(1 for r in results if r == -1)
processed = sum(1 for r in results if r > 0)
total_passages = sum(r for r in results if r > 0)

print(f"\nDone in {elapsed:.1f}s")
print(f"  Fragments processed: {processed}")
print(f"  Fragments skipped (already existed): {skipped}")
print(f"  Total passages produced: {total_passages:,}")

Segmenting 100 fragments on 24 workers
  Input:  C:\Users\Utente\Documents\PycharmProgetti\dl-RAG-denseAndKG\data\wikipedia_2018_clean\ordered_fragments
  Output: C:\Users\Utente\Documents\PycharmProgetti\dl-RAG-denseAndKG\data\wikipedia_2018_sentence_aligned\ordered_fragments


Segmenting: 100%|██████████| 100/100 [00:00<00:00, 274.62frag/s]


Done in 0.5s
  Fragments processed: 0
  Fragments skipped (already existed): 100
  Total passages produced: 0


In [8]:
import polars as pl
import time

# === Concatenate output fragments → final sentence-aligned corpus ===
#
# Each output fragment is a TSV with columns (text, title) — flat passages.
# We read all 100 in order, concatenate, assign sequential IDs (1-based to
# match DPR convention), and save as a single TSV.

SENTENCE_ALIGNED_DIR = PROJECT_ROOT / "data" / "wikipedia_2018_sentence_aligned"
SENTENCE_ALIGNED_PATH = SENTENCE_ALIGNED_DIR / "psgs_w100_sentence.tsv"

print("Concatenating output fragments...")
start = time.perf_counter()

# Read all fragments in order — guarantees passage ordering matches article ordering
frames = []
for i in range(N_FRAGS):
    frag_path = OUTPUT_FRAG_DIR / f"frag_{i}.tsv"
    assert frag_path.exists(), f"Missing output fragment: {frag_path}"
    frames.append(pl.read_csv(frag_path, separator="\t"))

final_df = pl.concat(frames)

# Free the list of individual DataFrames
del frames

# Add sequential IDs (1-based, matching DPR corpus convention)
# with_row_index adds a UInt32 column; offset=1 makes it 1-based
final_df = final_df.with_row_index("id", offset=1)

# Reorder columns to match DPR format: id, text, title
final_df = final_df.select("id", "text", "title")

# Save
final_df.write_csv(SENTENCE_ALIGNED_PATH, separator="\t")
elapsed = time.perf_counter() - start

size_gb = SENTENCE_ALIGNED_PATH.stat().st_size / (1024 ** 3)
print(f"\nSaved to: {SENTENCE_ALIGNED_PATH}")
print(f"Size: {size_gb:.1f} GB")
print(f"Rows: {len(final_df):,}")
print(f"Time: {elapsed:.1f}s")
print(f"\nSample:")
final_df.head(5)

Concatenating output fragments...

Saved to: C:\Users\Utente\Documents\PycharmProgetti\dl-RAG-denseAndKG\data\wikipedia_2018_sentence_aligned\psgs_w100_sentence.tsv
Size: 25.4 GB
Rows: 41,995,761
Time: 56.7s

Sample:


id,text,title
u32,str,str
1,"""Aaron Aaron ( or ; ""Ahärôn"") i…","""Aaron"""
2,"""Part of the Law (Torah) that M…","""Aaron"""
3,"""God at Sinai granted Aaron the…","""Aaron"""
4,"""At the command of Moses, he le…","""Aaron"""
5,"""his rod turn into a snake. The…","""Aaron"""


In [10]:
# === Verification ===

print("=" * 60)
print("VERIFICATION CHECKS")
print("=" * 60)

# 1. Total passage count — should be ~21M (similar to DPR original)
n_passages = len(final_df)
print(f"\n1. Total passages: {n_passages:,}")
print(f"   DPR original had ~21M passages for comparison")

# 2. Word count check — every passage should have exactly 100 words
#    Exception: articles shorter than 100 words (unpadded single segment)
sample = final_df.head(10000)
word_counts = sample["text"].str.count_matches(r" ") + 1
n_not_100 = word_counts.filter(word_counts != 100).len()
print(f"\n2. Word count check (first 10K passages):")
print(f"   Passages with exactly 100 words: {10000 - n_not_100}/10,000")
if n_not_100 > 0:
    print(f"   Passages with != 100 words: {n_not_100} (expected: only very short articles)")

# 3. PAEEK spot-check — known article, verify passages are well-formed
paeek_passages = final_df.filter(pl.col("title") == "PAEEK")
n_paeek = len(paeek_passages)
print(f"\n3. PAEEK spot-check:")
print(f"   Passages found: {n_paeek}")

for i, row in enumerate(paeek_passages.iter_rows(named=True)):
    wc = row["text"].count(" ") + 1
    words = row["text"].split(" ")
    print(f"   [{i}] {wc} words | start: {' '.join(words[:6])}...")

# 4. No articles lost — count distinct titles
n_titles = final_df["title"].n_unique()
print(f"\n4. Distinct article titles: {n_titles:,}")
print(f"   Original articles: {n_articles:,}")

# 5. Schema check
print(f"\n5. Schema: {dict(final_df.schema)}")
print(f"   Columns: {final_df.columns}")
assert final_df.columns == ["id", "text", "title"], "Column mismatch!"
print(f"   Matches expected format (id, text, title)")

print(f"\n{'=' * 60}")
print("ALL CHECKS PASSED")
print(f"{'=' * 60}")

VERIFICATION CHECKS

1. Total passages: 41,995,761
   DPR original had ~21M passages for comparison

2. Word count check (first 10K passages):
   Passages with exactly 100 words: 10000/10,000

3. PAEEK spot-check:
   Passages found: 8
   [0] 100 words | start: PAEEK PAEEK (; "Podosfairiki Athlitiki Enosi...
   [1] 100 words | start: The PAEEK was a founding member...
   [2] 100 words | start: Cyprus Basketball Federation in 1966 rising...
   [3] 100 words | start: They were knocked out by PAOK...
   [4] 100 words | start: teams as they were all refugees...
   [5] 100 words | start: For recent transfers see List of...
   [6] 100 words | start: symbolising the sadness from the Turkish...
   [7] 100 words | start: It now plays in exile in...

4. Distinct article titles: 3,237,506
   Original articles: 21,035,236

5. Schema: {'id': UInt32, 'text': String, 'title': String}
   Columns: ['id', 'text', 'title']
   Matches expected format (id, text, title)

ALL CHECKS PASSED
